In [111]:
import torch
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from torch import Tensor
from torch.nn import Sequential, Linear, ReLU, CrossEntropyLoss
from torch.utils.data import DataLoader, Dataset
from torch.optim import Adam

In [112]:
class MyDataset(Dataset):
    def __init__(self, datas: np.ndarray, labels: np.ndarray) -> None:
        self._x = torch.from_numpy(datas).float()
        self._y = torch.from_numpy(labels).long()

    def __len__(self) -> int:
        return len(self._x)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        int(idx)
        return self._x[idx], self._y[idx]

In [113]:
def train_model(features, targets, model, skf, batch, epochs, optimizer, criterion):
    acc = []
    f1 = []
    matriz = []
    for i, (train_index, test_index) in enumerate(skf.split(features, targets)):
        print(f"treinando model -------- fold {i + 1}")
        train_data = MyDataset(features[train_index], targets[train_index])
        train_loader = DataLoader(train_data, batch_size = batch)
        for i_epochs in range(epochs):
            model.train()
            for x, y in train_loader:
                optimizer.zero_grad()
                output = model(x)
                loss = criterion(output, y.squeeze().long()) #calcular meu erro com meu y verdadeiro
                loss.backward()
                optimizer.step()

        model.eval()
        with torch.no_grad():
            x_test_tensor = Tensor(features[test_index]).float()
            y_true = targets[test_index]

            result = model(x_test_tensor).cpu().numpy()
            y_pred = np.argmax(result, axis=1)

            acc_s = accuracy_score(y_true, y_pred)
            f1_s = f1_score(y_true, y_pred, average = "macro")
            matriz_s = confusion_matrix(y_true, y_pred)

            acc.append(acc_s)
            f1.append(f1_s)
            matriz.append(matriz_s)

            print(f"ACC: {acc_s}")
            print(f"F1_SCORE: {f1_s}")
            print(f"MATRIZ:{matriz_s}")

    print(f"\nResultado modelo ----- ACC: {np.mean(acc)}, F1_SCORE: {np.mean(f1)}, MATRIZ: {np.mean(matriz)}")


In [114]:
dataset = load_wine()
features = dataset.data
targets = np.expand_dims(dataset.target, axis=1)

In [115]:
model = Sequential(
    Linear(13,32),
    ReLU(),
    Linear(32,16),
    ReLU(),
    Linear(16,3)
)

In [116]:
optimizer = Adam(model.parameters(), lr=0.001)
criterion = CrossEntropyLoss()
max_epochs = 100
batch = 12
skf = StratifiedKFold(n_splits = 5, shuffle=True)

In [117]:
train_model(features, targets, model, skf, batch, max_epochs, optimizer, criterion)

treinando model -------- fold 1
ACC: 0.9166666666666666
F1_SCORE: 0.9162121448236454
MATRIZ:[[11  1  0]
 [ 0 14  0]
 [ 0  2  8]]
treinando model -------- fold 2
ACC: 0.9166666666666666
F1_SCORE: 0.9201228878648232
MATRIZ:[[ 9  3  0]
 [ 0 14  0]
 [ 0  0 10]]
treinando model -------- fold 3
ACC: 0.9166666666666666
F1_SCORE: 0.9193411952032641
MATRIZ:[[10  2  0]
 [ 0 13  1]
 [ 0  0 10]]
treinando model -------- fold 4
ACC: 0.9142857142857143
F1_SCORE: 0.9176413255360623
MATRIZ:[[11  1  0]
 [ 1 12  1]
 [ 0  0  9]]
treinando model -------- fold 5
ACC: 0.9714285714285714
F1_SCORE: 0.9733742959549412
MATRIZ:[[10  1  0]
 [ 0 15  0]
 [ 0  0  9]]

Resultado modelo ----- ACC: 0.927142857142857, F1_SCORE: 0.9293383698765473, MATRIZ: 3.9555555555555557
